# The exposure regression: rolling loadings and the factor split

*A learning exercise performed in role: a simulated mandate with no client and no institution. Nothing in this notebook is investment advice, a recommendation, or a client communication.*

**Entry point** `python3 -m portfolio_workbench.factors.exposures`

**Modules covered** `factors/exposures.py`

_Generated from the code by `python3 -m reporting.notebooks`: the module headers below are read out of the modules themselves, and the run is the entry point's own output._

## 1. What this module does, and the source of every method in it

The exposure layer fits every sleeve on the named set, once per estimation window, and splits each sleeve's variance into the part its factors explain and the part they do not. The entry point prints the loadings, the identity check on the four constructed sleeves, the alphas with their t-statistics, and the factor and idiosyncratic shares of tracking error.

**Sources.** Every public function of the modules this notebook covers, and what it traces to. The map is checked over the code by the acceptance fixture, so a method added without a source fails a command rather than going unnoticed.

- `factors/exposures.py::conditioning` traces to the exposure regression of this effort: ordinary least squares per window on the named spine plus the constructed block, orthogonalised within the window so the block is net of the spine it is built from
- `factors/exposures.py::expanding_windows` traces to the exposure regression of this effort: ordinary least squares per window on the named spine plus the constructed block, orthogonalised within the window so the block is net of the spine it is built from
- `factors/exposures.py::factor_split` traces to the exposure regression of this effort: ordinary least squares per window on the named spine plus the constructed block, orthogonalised within the window so the block is net of the spine it is built from
- `factors/exposures.py::fixed_alpha` traces to the fixed-alpha restriction as a diagnostic, reported because a panel of diversified index legs is expected to price near beta one on its own benchmark
- `factors/exposures.py::loading_drift` traces to the exposure regression of this effort: ordinary least squares per window on the named spine plus the constructed block, orthogonalised within the window so the block is net of the spine it is built from
- `factors/exposures.py::main` traces to the exposure regression of this effort: ordinary least squares per window on the named spine plus the constructed block, orthogonalised within the window so the block is net of the spine it is built from
- `factors/exposures.py::orthogonal_transform` traces to the exposure regression of this effort: ordinary least squares per window on the named spine plus the constructed block, orthogonalised within the window so the block is net of the spine it is built from
- `factors/exposures.py::orthogonalise` traces to the block orthogonalised against the spine inside the same window, so a block loading is the sleeve's construction and not a fit
- `factors/exposures.py::regress` traces to ordinary least squares per window with the residual split reported: the factor and idiosyncratic parts of the tracking error
- `factors/exposures.py::rolling` traces to the exposure regression of this effort: ordinary least squares per window on the named spine plus the constructed block, orthogonalised within the window so the block is net of the spine it is built from
- `factors/exposures.py::variance_inflation` traces to the variance inflation factor, the diagnostic for the collinearity the orthogonalisation exists to remove
- `factors/exposures.py::window_block` traces to the exposure regression of this effort: ordinary least squares per window on the named spine plus the constructed block, orthogonalised within the window so the block is net of the spine it is built from
- `factors/exposures.py::windows` traces to the exposure regression of this effort: ordinary least squares per window on the named spine plus the constructed block, orthogonalised within the window so the block is net of the spine it is built from
- `factors/exposures.py::within_span` traces to the exposure regression of this effort: ordinary least squares per window on the named spine plus the constructed block, orthogonalised within the window so the block is net of the spine it is built from

## 2. Why it works this way, including what was rejected

_The module headers, verbatim: each records why the module is shaped the way it is, what was rejected, and the measurement that settled it. They are quoted here rather than restated, so the notebook cannot drift from the code._

**`factors/exposures.py`**

Rolling exposures, alpha, and the factor/idiosyncratic split of return variance.

Every sleeve's EUR excess return is regressed on the named factor set in a trailing window, and the
window is refitted every month. Five things in here are decisions rather than mechanics, and the
first two exist because the loadings are not identified without them.

**The window is stated in month labels, not in return observations.** It is the sixty months ending
in the month before the traded month, which is all the availability rule allows, since a bar for
month M is readable from the first day of M+1. The calendar the windows are cut on is the panel's
own, so the out-of-sample run keeps its decided length; the panel's first bar has no return, so the
earliest window carries fifty-nine observations where later ones carry sixty, and the count prints
per window rather than being assumed.

**The block is orthogonalised against itself inside each window.** Its four declared series are built
from four sleeves that move together: the level and the slope correlate 0.997 or higher in every
window on this panel, because the short government sleeve is far the quieter of the two, and the
credit spread sits at -0.96 against the level. A coefficient on either half of a pair like that is
not identified: the worst loading's variance is inflated by a factor of hundreds to thousands, where a
design whose regressors were unrelated would give one. Projecting the block onto itself in its declared order
leaves the level exactly as declared and gives the later series the cleaner reading - the slope net
of the level, the credit spread net of the term structure, high yield net of credit - with the
design's conditioning back at the spine's own. Nothing is centred, because a monthly factor return's
mean belongs to the factor model: centring would move the intercept and turn the reported alpha from
Jensen's into the sleeve's own average return. The transform is derived from the window alone, since
everything a window uses is inside it, and the fitted values, alphas, residual variances and shares
of variance are **identical** under either parameterisation - only the loadings and their standard
errors change.

**Four sleeves are the block, so the block is dropped from their design.** `IBGL.AS`, `IEGE.AS`,
`IEAC.AS` and `IHYG.L` are exact linear combinations of the block - `IBGL = level + slope/2`,
`IEGE = level - slope/2`, `IEAC = level + slope/2 + credit`, and `IHYG` adds the high-yield excess -
because the block is built from them. Regressing such a sleeve on a set containing its own
construction fits it exactly, reports every other loading as zero, and destroys the one exposure that
*is* estimable: its sensitivity to the published spine. Those sleeves are therefore regressed on the
spine alone, with the block part reported as the identity it is. Their full-model alpha is zero by
arithmetic rather than by evidence, so no alpha is reported for them, and their mean excess return
decomposes exactly into its spine part and its block part.

**Loadings roll, and a fixed-loading run is kept as the contrast.** The study is about estimation
error, and fixed loadings would hide the quantity under study; the size of the difference between the
two runs is itself the measurement of loading drift.

**The variance split is exact for variance, not for volatility.** With an intercept the fitted and
residual sums of squares partition the total exactly, so the two parts add to the sleeve's own
variance to machine precision. Volatility is the square root of each part and does not add, which is
why shares of variance are what the report prints.

## 3. The data contract it consumes, and the as-of rule

One ordinary least squares fit per window per group of sleeves, with an intercept, on the factors as declared. The window is sixty months by decision and the first window is allowed to be shorter than the rest only where the panel begins, which is stated on the step rather than smoothed. Every frame is reindexed onto the fit's own traded months, so a month cannot enter a regression the window did not cover, and the residual split divides by the residual degrees of freedom so that the fitted and residual parts sum to the sleeve's own sample variance.

## 4. The worked example on small numbers, with the identity checked

Ordinary least squares on a window with two factors and five months, where the series is generated from known loadings: the recovered coefficients are the planted ones, which is the check that the design matrix, the alignment and the residual split are the arithmetic they claim.

The cell below runs on numbers small enough to check by hand and asserts the identity, so a reader can see the arithmetic rather than take the module's word for it.

In [1]:
import numpy as np
import pandas as pd

from portfolio_workbench.factors import exposures

months = pd.PeriodIndex(["2020-01", "2020-02", "2020-03", "2020-04", "2020-05"], freq="M")
factors = pd.DataFrame({"market": [0.01, -0.02, 0.03, 0.00, 0.02], "credit": [0.005, 0.001, -0.004, 0.002, 0.0]}, index=months)
# A sleeve planted as 2 x market + 3 x credit + a constant, with no residual at all.
sleeve = 2.0 * factors["market"] + 3.0 * factors["credit"] + 0.001
fit = exposures.regress(pd.DataFrame({"sleeve": sleeve}), factors)

assert np.allclose(np.asarray(fit["beta"]).ravel(), [2.0, 3.0], atol=1e-12)
assert abs(float(np.asarray(fit["alpha"]).ravel()[0]) - 0.001) < 1e-12
assert float(np.asarray(fit["r2"]).ravel()[0]) == 1.0
print("recovered loadings:", np.round(np.asarray(fit["beta"]).ravel(), 12).tolist())

recovered loadings: [2.0, 3.0]


## 5. The real run: inputs, parameters, provenance block

The provenance block is printed first, then the parameters this module decides under, then the entry point's own report. The report is the module's output rather than a transcription of it, so a number quoted from a notebook is the number the module prints.

In [2]:
from portfolio_workbench.data import loader, universe

document = loader.load_panel()
months = document.months
print(f"snapshot {document.snapshot_id}, taken as of {document.as_of}")
print(f"panel {len(months)} months {months.min()}..{months.max()} across {len(universe.TICKERS)} sleeves")
print("manifest fields: " + ", ".join(sorted(document.manifest)))

from portfolio_workbench.factors import exposures

print(f"estimation window {exposures.WINDOW} months, minimum observations {exposures.MIN_OBS}")
print(f"block: {', '.join(exposures.BLOCK)}")

snapshot 2026-09-13, taken as of 2026-09-13T07:56:28+00:00
panel 191 months 2010-09..2026-07 across 11 sleeves
manifest fields: created, excluded, files, instruments, snapshot_id, window
estimation window 60 months, minimum observations 59
block: government_level, term_slope, credit, high_yield_excess


In [3]:
import subprocess
import sys

finished = subprocess.run(
    [sys.executable, "-m", "portfolio_workbench.factors.exposures"], capture_output=True, text=True, cwd="."
)
print(finished.stdout)
assert finished.returncode == 0, finished.stderr

[factor] snapshot 2026-09-13, 10 factors, 131 refits 2015-09..2026-07 on the panel calendar 2010-09..2026-07
[factor] window 60 month labels ending the month before the traded month; the first window 2010-09..2015-08 carries 59 observations (the panel's first bar has no return), later windows 60
[factor] the block is orthogonalised against itself inside each window: as declared the level and the slope correlate 0.997..1.000, and the worst loading's variance is inflated x309..x2,889 by the declared block - a coefficient on the level or the slope is not identified there - against x8..x32 once it is orthogonalised, with x8..x31 for the spine alone. What is left is the published factors' own correlation, which is estimable; fitted values, alphas and variance shares are identical either way, and only the loadings and their standard errors change
[factor] orthogonalising leaves the first series alone and makes the later ones quieter, so a block loading is per unit of that series: declared vo

## 6. Results, and how to read them, including the resolution limit and what a reader must not conclude

The R-squared on the four construction sleeves is one by construction, so it is a check on the arithmetic and not a result; the seven other sleeves carry the interpretable alphas. At sixty-month windows and eleven sleeves the loadings are estimated with error, and the layer reports the variance inflation and the loading drift beside them rather than claiming stability. A reader must not read a non-zero alpha on a diversified index leg as skill: an index sleeve's alpha is a statement about the factor set's ability to price that market, and the layer says as much.

## 7. What this module does not establish

Nothing here establishes that the factors are priced, that the loadings are constant outside the window, or that a low alpha means a well-priced market. The regression measures association within windows, and its fixed-alpha and variance-inflation diagnostics are reported rather than acted on.